# v26 MaxViT-Tiny — Ensemble Teacher Notebook

**Companion to v26_final (EfficientNet-B4).** Trains a diverse architecture
(MaxViT-Tiny: CNN + multi-axis global attention hybrid, ImageNet-21k pretrain)
on the SAME corrected labels, SAME folds, SAME holdout (SEED=42) as the B4 notebook.

## Purpose
Produce per-fold P2 test predictions (`test_lb_maxvit_p2_fold{0-4}.npy`) to use as a
50/50 ensemble teacher in the B4 v26 PL continuation. Architecture diversity →
better pseudo-labels → less overfit → higher final B4 F1.

## What this notebook is NOT
- NO pseudo-label training (PL) — that happens in the B4 continuation
- NO ONNX export — the final eval model stays B4 (fast, single-pass)
- NO GAIN, NO Gabor dual-stream — those are B4-specific; MaxViT gets diversity from its architecture

## Pipeline (P1 → P2 only)
1. Priority-1 rule label correction (identical to B4 — deterministic)
2. bottletypes.csv + joint-stratified folds (identical SEED=42)
3. 10% holdout (identical split to B4 → comparable)
4. P1: 5 epochs, fresh 21k-pretrained init, curriculum bucket-A, SWA
5. (Re-run cleanlab after P1 for informed triage — optional)
6. P2: 4 epochs, fine-tune on cleaned labels, SWA
7. Teacher export: per-fold + mean P2 test predictions saved as .npy

## Setup
Add Data → competition data (must include bottletypes.csv). Internet ON (timm pulls
MaxViT-Tiny weights from HuggingFace). GPU T4. Then Run All.

## Runtime
~11h on T4 (BS=12, grad-accum 3, grad-checkpoint ON). Crash-safe: re-running skips
completed folds. If a session times out mid-P1, Save Version and re-run — it resumes.

## After it finishes
Save Version → make a Kaggle Dataset from the output `.npy` files → attach to the
B4 v26 PL-continuation notebook. Its PL-gen cell auto-detects `*maxvit*` files.


# Local Mac Runtime Notes

This leaderboard copy was generated for Apple Silicon local runs. It keeps the competition logic intact while using project-relative paths, `mps` device detection, and Mac-friendly memory defaults.


# Leaderboard Ensemble Member: MaxViT Tiny

- `VERSION_TAG`: `lb_maxvit`
- `MODEL_NAME`: `maxvit_rmlp_tiny_rw_256.sw_in1k`
- Outputs: `outputs/lb_maxvit/`


# Leaderboard Mode: MaxViT Tiny

This notebook disables the extra 10% holdout and uses all labeled images in 5-fold CV training.
- `VERSION_TAG`: `lb_maxvit`
- `MODEL_NAME`: `maxvit_rmlp_tiny_rw_256.sw_in1k`
- Outputs: `outputs/lb_maxvit/`


In [ ]:
# =========================================================
# CELL 1: Local dependency check
# =========================================================
# Install packages from the terminal before running this notebook:
#   source .venv/bin/activate
#   uv pip install -r requirements-mac.txt

import importlib.util

required_modules = [
    'torch', 'torchvision', 'timm', 'albumentations', 'cv2',
    'numpy', 'pandas', 'sklearn', 'scipy', 'tqdm', 'cleanlab'
]

missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError(
        'Missing packages: ' + ', '.join(missing) +
        '. From the project root, run: uv pip install -r requirements-mac.txt'
    )

print('Local dependency check done.')


In [ ]:
# =========================================================
# CELL 2: Imports + seeds + device
# =========================================================
import os
# Set before importing torch. MPS fallback lets unsupported ops run on CPU.
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
os.environ.setdefault('PYTORCH_MPS_HIGH_WATERMARK_RATIO', '0.0')

import gc, json, time, random, warnings, math, copy
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import scipy.ndimage as ndi

warnings.filterwarnings('ignore')

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
set_seed()

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

PRIMARY_DEVICE = DEVICE
AMP_ENABLED = DEVICE.type == 'cuda'
AMP_DEVICE_TYPE = 'cuda' if AMP_ENABLED else 'cpu'
PIN_MEMORY = DEVICE.type == 'cuda'

def amp_autocast(enabled=None):
    return torch.amp.autocast(
        device_type=AMP_DEVICE_TYPE,
        enabled=AMP_ENABLED if enabled is None else enabled,
    )

def make_grad_scaler():
    try:
        return torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

def empty_device_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif hasattr(torch, 'mps') and hasattr(torch.mps, 'empty_cache'):
        torch.mps.empty_cache()

print(f'PyTorch {torch.__version__} | timm {timm.__version__} | device {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
elif DEVICE.type == 'mps':
    print('Apple Silicon MPS backend active')
else:
    print('CPU backend active')


In [ ]:
# =========================================================
# CELL 3: Config — MaxViT-Tiny (ensemble teacher for B4 v26)
# =========================================================
# Purpose: train a DIVERSE architecture (MaxViT = CNN+global attention hybrid)
# on the SAME corrected labels + SAME folds + SAME holdout as v26 B4.
# Output: per-fold P2 test predictions (.npy) → used as ensemble teacher in
# the B4 v26 PL continuation. This notebook does NOT do PL or export ONNX.
VERSION_TAG = 'lb_maxvit'
MODEL_NAME  = 'maxvit_rmlp_tiny_rw_256.sw_in1k'
# Architecture diversity vs B4: MaxViT = CNN + window/grid global attention hybrid (very different inductive bias)

# ── Local paths ───────────────────────────────────────────────
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for cand in candidates:
        if (cand / 'requirements-mac.txt').exists() and (cand / 'notebooks').exists():
            return cand
    return start

PROJECT_ROOT = find_project_root()
DATA_CANDIDATES = [
    PROJECT_ROOT / 'data' / '1st-krones-vision-ai-challenge',
    PROJECT_ROOT / 'data',
    PROJECT_ROOT,
    Path.home() / 'Downloads' / '1st-krones-vision-ai-challenge',
]
DATA = None
for cand in DATA_CANDIDATES:
    if (cand / 'train.csv').exists() and (cand / 'train_annotations.json').exists():
        DATA = cand
        break
if DATA is None:
    raise FileNotFoundError(
        'Dataset not found. Extract it to '
        f"{PROJECT_ROOT / 'data' / '1st-krones-vision-ai-challenge'}"
    )
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA: {DATA}')
SAVE = PROJECT_ROOT / 'outputs' / VERSION_TAG
SAVE.mkdir(parents=True, exist_ok=True)

# ── Local Mac memory budget ───────────────────────────────────
# Start conservatively on 16 GB unified memory. Raise BATCH_SIZE if stable.
BATCH_SIZE          = 4
GRAD_ACCUM_STEPS    = 8      # effective batch = 32 (matches original)
USE_GRAD_CHECKPOINT = True

# ── Training hyperparameters ─────────────────────────────────
IMG_SIZE       = 256         # native res for maxvit_rmlp_tiny_rw_256 — ~2.25x faster than 384
NUM_WORKERS    = 0          # safer inside local Jupyter on macOS
DROPOUT        = 0.3         # MaxViT regularizes well already; lighter dropout
N_FOLDS        = 5
WEIGHT_DECAY   = 1e-4

# Phase epochs — MaxViT converges fast from 21k pretrain. P1+P2 only (NO PL here).
P1_EPOCHS      = 5
P2_EPOCHS      = 4
LR_P1          = 2e-4        # slightly lower than B4 — transformers prefer lower LR
LR_P2          = 8e-5

# Stochastic depth (drop_path_rate)
DROP_PATH_RATE_P1  = 0.2

# SWA
SWA_START_FRAC = 0.6
SWA_LR         = 1e-5

# ── TTA — 6 modes (no vflip, no 90/180°) ────────────────────
TTA_MODES = ['none', 'hflip', 'rot5', 'rot-5', 'scale09', 'scale11']

# ── Hold-out split (MUST match v26 B4: same SEED, same fraction) ──
HOLDOUT_FRACTION = 0.0

# ── GAIN / Gabor — DISABLED (B4-specific, no value on MaxViT) ──
USE_GAIN          = False
USE_GABOR_STREAM  = False
GAIN_OMEGA        = 0.0
GAIN_FEAT_SIZE    = 12
GAIN_MASK_SIGMA   = 3
GABOR_ORIENTATIONS = [0, 45, 90, 135]
GABOR_OUT_CHANNELS = 8

# ── Cleanlab confident learning + curriculum ─────────────────
USE_CLEANLAB    = True
NLS_ALPHA       = -0.1
BUCKET_B_WEIGHT = 0.3
BUCKET_C_DROP_CONF = 0.90
BUCKET_B_FLAG_CONF = 0.70

# ── Consistency (unused in MaxViT P1/P2, but train_fold references it) ──
CONSISTENCY_WEIGHT = 0.0

# ── MC-Dropout uncertainty (kept for parity, light) ──────────
MC_T              = 30
MC_UNCERTAIN_TOP  = 0.05

# ── Bottle types ─────────────────────────────────────────────
BOTTLE_TYPE_CANONICAL = {'VICHY': 0, 'EURO': 1, 'NRW': 2}
N_BOTTLE_TYPES = 3

# ── Noise cleaning ───────────────────────────────────────────
NOISE_CONF_THRESH = 0.95

# ── FiftyOne ─────────────────────────────────────────────────
USE_FIFTYONE = False

# ── Priority 1 label correction ──────────────────────────────
LABEL_CORRECTION_ENABLED = True

# Official COCO target rules from the competition brief.
ALWAYS_FAULTY_CATS = {
    'Break / Crack',
    'Circlip',
    'Contamination dark',
    'Crown cap',
    'Foil / Semitransparent',
    'Foreign object - manual cleaning',
    'Foreign object - washing machine',
    'Glass shard',
    'Insect',
    'Label',
    'Liquid',
    'Mold',
    'No base visible',
    'Paint residue',
    'Straw',
    'Yeast residue',
}
CONDITIONAL_THRESHOLDS = {
    'Air bubble': 500,
    'Chip': 200,
    'Contamination light': 180,
    'Glass imperfection': 100,
    'Scuffing': 75000,
    'Scuffing heavy': 1200,
}
ALWAYS_GOOD_CATS = {
    'Embossing',
    'Foam residue',
    'No fault',
    'Water drop',
    'ROI',
    'Roi',
    'roi',
    'Region of interest',
    'background',
    'Background',
}


print(f'\nModel config: {VERSION_TAG}')
print(f'  Backbone: {MODEL_NAME}')
print(f'  IMG_SIZE={IMG_SIZE}, BS={BATCH_SIZE} (accum×{GRAD_ACCUM_STEPS}=eff {BATCH_SIZE*GRAD_ACCUM_STEPS}), folds={N_FOLDS}')
print(f'  Phases: P1({P1_EPOCHS}ep) + P2({P2_EPOCHS}ep) — NO PL, NO ONNX (teacher-only)')
print(f'  GAIN/Gabor DISABLED (B4-specific). Cleanlab={USE_CLEANLAB}, GradCkpt={USE_GRAD_CHECKPOINT}')
print(f'  Holdout: {HOLDOUT_FRACTION*100:.0f}% (SAME SEED={SEED} as v26 B4 → comparable)')


In [ ]:
# =========================================================
# CELL 4: 🆕 DATA AUDIT — image data analysis before model training
# =========================================================
# Surfaces everything the pipeline knows about the data before a single
# gradient is computed. Spot class imbalance, hard categories, ROI issues
# upfront so configuration can be sanity-checked.

print('═══════════════════════════════════════════════════════════════')
print('  DATA AUDIT — 1st Krones Vision AI Challenge')
print('═══════════════════════════════════════════════════════════════')

# --- train.csv ---
FINAL_TRAIN_CSV = PROJECT_ROOT / 'outputs' / 'preprocessing' / 'final_train.csv'
if FINAL_TRAIN_CSV.exists():
    labels_orig = pd.read_csv(FINAL_TRAIN_CSV)
    print(f'Using COCO-derived training labels: {FINAL_TRAIN_CSV}')
else:
    labels_orig = pd.read_csv(DATA / 'train.csv')
    print('COCO final_train.csv not found yet; this notebook will derive corrected labels from annotations.')
n_total = len(labels_orig)
n_class1 = int(labels_orig.target.sum())
n_class0 = n_total - n_class1
print(f'\nTRAIN.CSV')
print(f'  Total images: {n_total}')
print(f'  Class 0 (reusable): {n_class0}  ({n_class0/n_total*100:.2f}%)')
print(f'  Class 1 (faulty):   {n_class1}  ({n_class1/n_total*100:.2f}%)')

# --- bottletypes.csv ---
BTYPES_PATH = DATA / 'bottletypes.csv'
if not BTYPES_PATH.exists():
    for cand in [PROJECT_ROOT / 'data' / 'bottletypes' / 'bottletypes.csv']:
        if cand.exists(): BTYPES_PATH = cand; break
assert BTYPES_PATH.exists(), 'bottletypes.csv not found'
bt_df = pd.read_csv(BTYPES_PATH)

def canonical_bottle_type(s):
    sl = str(s).lower()
    if 'vichy' in sl: return 0
    if 'euro'  in sl: return 1
    if 'nrw'   in sl: return 2
    return -1
bt_df['btype_id'] = bt_df['bottle_type'].apply(canonical_bottle_type)
bt_df['stem']     = bt_df['image_id'].apply(lambda x: Path(str(x)).stem)
btype_map = dict(zip(bt_df['stem'], bt_df['btype_id']))

print(f'\nBOTTLETYPES.CSV')
print(f'  Total entries: {len(bt_df)}  | Unique strings: {bt_df.bottle_type.nunique()}')
print(f'  Split: {bt_df["split"].value_counts().to_dict()}')
print(f'  Unmapped: {(bt_df.btype_id == -1).sum()}')
print('\n  Per bottle type (train/test):')
print(f'  {"":7s}  {"TRAIN":>7s}  {"TEST":>7s}')
for tname, tid in BOTTLE_TYPE_CANONICAL.items():
    n_tr = ((bt_df.btype_id == tid) & (bt_df['split'] == 'train')).sum()
    n_te = ((bt_df.btype_id == tid) & (bt_df['split'] == 'test')).sum()
    print(f'  {tname:7s}  {n_tr:>7d}  {n_te:>7d}')

# --- Per-bottle-type class-1 rate (THE critical insight) ---
labels_orig['stem']     = labels_orig['image_id'].apply(lambda x: Path(str(x)).stem)
labels_orig['btype_id'] = labels_orig['stem'].map(btype_map).fillna(-1).astype(int)
print('\n  Per-type class-1 rate (training):')
for tname, tid in BOTTLE_TYPE_CANONICAL.items():
    sub = labels_orig[labels_orig.btype_id == tid]
    if len(sub) > 0:
        rate = sub.target.mean()
        print(f'    {tname:7s}: n={len(sub):>5d}  class1_rate={rate:.4f}  '
              f'(faulty={int(sub.target.sum())}, reusable={int(len(sub)-sub.target.sum())})')

# --- train_annotations.json ---
print(f'\nTRAIN_ANNOTATIONS.JSON')
with open(DATA / 'train_annotations.json') as f:
    train_ann_data = json.load(f)
n_imgs_ann = len(train_ann_data['images'])
n_anns     = len(train_ann_data['annotations'])
n_cats     = len(train_ann_data['categories'])
print(f'  Images with annotations: {n_imgs_ann}')
print(f'  Total annotations: {n_anns}')
print(f'  Categories: {n_cats}')

# Category counts
cat_id_to_name = {c['id']: c['name'] for c in train_ann_data['categories']}
cat_counts = Counter()
for ann in train_ann_data['annotations']:
    cat_counts[cat_id_to_name.get(ann['category_id'], 'unknown')] += 1

print('\n  Category distribution (annotations per category, sorted desc):')
for cname, count in cat_counts.most_common():
    tag = 'FAULTY' if cname in ALWAYS_FAULTY_CATS else \
          ('COND' if cname in CONDITIONAL_THRESHOLDS else
           ('GOOD' if cname in ALWAYS_GOOD_CATS else 'UNKNOWN'))
    print(f'    [{tag:7s}] {cname:50s} {count:>6d}')

# Area statistics for conditional categories
print('\n  Conditional category area statistics (need threshold comparison):')
cat_areas = defaultdict(list)
for ann in train_ann_data['annotations']:
    cn = cat_id_to_name.get(ann['category_id'], '')
    if cn in CONDITIONAL_THRESHOLDS:
        cat_areas[cn].append(float(ann.get('area', 0)))
for cn, thr in CONDITIONAL_THRESHOLDS.items():
    if cn in cat_areas and len(cat_areas[cn]) > 0:
        a = np.array(cat_areas[cn])
        n_above = int((a >= thr).sum())
        print(f'    {cn:25s}  thr={thr:>6d}  n={len(a):>5d}  '
              f'median={np.median(a):>7.0f}  ≥thr={n_above:>5d} '
              f'({n_above/len(a)*100:.1f}%)')

# --- test_annotations_roi_only.json ---
print(f'\nTEST_ANNOTATIONS_ROI_ONLY.JSON')
test_ann_path = None
for cand in [DATA/'test_annotations_roi_only.json', DATA/'test_annotations.json']:
    if cand.exists(): test_ann_path = cand; break
if test_ann_path:
    with open(test_ann_path) as f:
        test_ann_data = json.load(f)
    n_test_imgs = len(test_ann_data['images'])
    n_test_anns = len(test_ann_data['annotations'])
    print(f'  File: {test_ann_path.name}')
    print(f'  Test images: {n_test_imgs}')
    print(f'  Test ROI annotations: {n_test_anns}')
    coverage = n_test_anns / max(1, n_test_imgs) * 100
    print(f'  ROI coverage: {coverage:.1f}%')
else:
    print('  ⚠️ No test annotations file found')
    test_ann_data = None

# --- Multi-category images (hard cases) ---
print('\nHARD CASE ANALYSIS')
img_id_to_stem = {img['id']: Path(img['file_name']).stem for img in train_ann_data['images']}
img_cats = defaultdict(set)
for ann in train_ann_data['annotations']:
    cn = cat_id_to_name.get(ann['category_id'], '')
    if cn in ALWAYS_GOOD_CATS: continue
    st = img_id_to_stem.get(ann['image_id'])
    if st: img_cats[st].add(cn)

multi_cat = {st: cats for st, cats in img_cats.items() if len(cats) > 1}
print(f'  Images with ≥2 defect categories: {len(multi_cat)}')
if len(multi_cat):
    multi_pairs = Counter()
    for st, cats in multi_cat.items():
        if len(cats) == 2:
            multi_pairs[tuple(sorted(cats))] += 1
    print('  Top co-occurring category pairs:')
    for pair, cnt in multi_pairs.most_common(5):
        print(f'    {pair[0]} + {pair[1]}: {cnt}')

print('\n═══════════════════════════════════════════════════════════════')
print('  AUDIT COMPLETE — proceed to ROI analysis')
print('═══════════════════════════════════════════════════════════════')


In [ ]:
# =========================================================
# CELL 5: 🆕 ROI ANALYSIS — verify host's claim "most ROIs ~512x512"
# =========================================================
# The host stated: "Most ROIs are conveniently close to 512x512 px."
# Confirm this empirically AND check per-bottle-type ROI sizes — this is the
# reason v23's ROI-area-inferred bottle types failed (all ROIs same size).

print('═══════════════════════════════════════════════════════════════')
print('  ROI ANALYSIS')
print('═══════════════════════════════════════════════════════════════')

# Build ROI map from train annotations
def find_roi_cat_ids(ann_data):
    return {c['id'] for c in ann_data.get('categories', [])
            if c.get('name', '').strip().lower() in {'roi', 'region of interest'}}

def build_roi_map(ann_data):
    if ann_data is None: return {}
    id_to_stem = {img['id']: Path(img['file_name']).stem for img in ann_data['images']}
    roi_ids = find_roi_cat_ids(ann_data) or {22}  # fallback to category 22
    return {id_to_stem[a['image_id']]: a['bbox']
            for a in ann_data['annotations']
            if a['category_id'] in roi_ids and a['image_id'] in id_to_stem}

TRAIN_ROI = build_roi_map(train_ann_data)
TEST_ROI  = build_roi_map(test_ann_data) if test_ann_data else {}
print(f'\nTRAIN_ROI: {len(TRAIN_ROI)} bboxes')
print(f'TEST_ROI:  {len(TEST_ROI)} bboxes')

# ROI size distribution
if len(TRAIN_ROI) > 0:
    widths  = np.array([bbox[2] for bbox in TRAIN_ROI.values()])
    heights = np.array([bbox[3] for bbox in TRAIN_ROI.values()])
    areas   = widths * heights
    print('\nTRAIN ROI dimensions:')
    print(f'  Width:  min={widths.min():.0f}  median={np.median(widths):.0f}  max={widths.max():.0f}  std={widths.std():.0f}')
    print(f'  Height: min={heights.min():.0f}  median={np.median(heights):.0f}  max={heights.max():.0f}  std={heights.std():.0f}')
    print(f'  Area:   min={areas.min():.0f}  median={np.median(areas):.0f}  max={areas.max():.0f}')
    near_512 = ((widths >= 500) & (widths <= 524) & (heights >= 500) & (heights <= 524)).sum()
    print(f'  ROIs within ±12px of 512×512: {near_512}/{len(widths)} ({near_512/len(widths)*100:.1f}%)')

# Per-type ROI size (confirms v23's ROI-area inference was broken)
print('\nPer-bottle-type ROI sizes:')
print(f'  {"":7s}  {"n":>5s}  {"w_med":>7s}  {"h_med":>7s}  {"area_med":>9s}')
for tname, tid in BOTTLE_TYPE_CANONICAL.items():
    stems_in_type = [s for s, t in btype_map.items() if t == tid]
    type_widths, type_heights = [], []
    for s in stems_in_type:
        if s in TRAIN_ROI:
            type_widths.append(TRAIN_ROI[s][2])
            type_heights.append(TRAIN_ROI[s][3])
    if type_widths:
        wm = np.median(type_widths); hm = np.median(type_heights)
        am = wm * hm
        print(f'  {tname:7s}  {len(type_widths):>5d}  '
              f'{wm:>7.0f}  {hm:>7.0f}  {am:>9.0f}')

# Test ROI same check
if len(TEST_ROI) > 0:
    twidths  = np.array([b[2] for b in TEST_ROI.values()])
    theights = np.array([b[3] for b in TEST_ROI.values()])
    print(f'\nTEST ROI dimensions:')
    print(f'  Width  median: {np.median(twidths):.0f}')
    print(f'  Height median: {np.median(theights):.0f}')

# Missing ROI check
labels_orig_stems = set(labels_orig['stem'])
train_with_roi    = set(TRAIN_ROI.keys())
missing_roi = labels_orig_stems - train_with_roi
if missing_roi:
    print(f'\n⚠️ {len(missing_roi)} training images missing ROI annotation')
else:
    print(f'\n✅ All {len(labels_orig_stems)} training images have ROI')

print('\n═══════════════════════════════════════════════════════════════')

# Test IDs for inference
if (DATA / 'sample_submission.csv').exists():
    test_df_init = pd.read_csv(DATA / 'sample_submission.csv')
    test_ids = test_df_init['image_id'].astype(str).tolist()
elif test_ann_data:
    test_ids = [img['file_name'] for img in test_ann_data['images']]
else:
    raise RuntimeError('No way to derive test_ids')
print(f'test_ids: {len(test_ids)} from {"sample_submission.csv" if (DATA/"sample_submission.csv").exists() else "test annotations"}')


In [ ]:
# =========================================================
# CELL 6: FiftyOne advanced workflows (OPTIONAL — local only)
# =========================================================
# Per research doc 1 — advanced label-noise + similarity workflows.
# Set USE_FIFTYONE=True in config (cell 3) to run locally.
# Outputs a CSV of flagged samples for use by cleanlab cell (cell 10).
#
# Workflows:
#   1. compute_mistakenness — finds confident-but-wrong samples
#   2. compute_similarity (CLIP) — finds visually similar edge cases
#   3. compute_uniqueness — diversity-aware sampling
#   4. Hard-label filtered views — Insect, Break/Crack, etc.

if USE_FIFTYONE:
    try:
        import fiftyone as fo
        import fiftyone.brain as fob
        from fiftyone import ViewField as F

        DS_NAME = f'krones_{VERSION_TAG}'
        if DS_NAME in fo.list_datasets():
            fo.delete_dataset(DS_NAME)
        dataset = fo.Dataset.from_dir(
            dataset_type=fo.types.COCODetectionDataset,
            data_path=str(DATA / 'train_images'),
            labels_path=str(DATA / 'train_annotations.json'),
            name=DS_NAME,
            label_types=['detections', 'segmentations'],
        )
        print(f'FiftyOne loaded {len(dataset)} samples')

        # Workflow 2: CLIP similarity (works without OOF)
        fob.compute_similarity(dataset, model='clip-vit-base32-torch',
                               brain_key='clip_sim')
        # Workflow 3: uniqueness
        fob.compute_uniqueness(dataset)
        # Workflow 4: hard-label views
        for cat in ['Insect', 'Break/Crack', 'Glass shard',
                    'Foreign object - manual cleaning']:
            v = dataset.filter_labels('ground_truth',
                                      F('label').is_in([cat]))
            print(f'  {cat}: {len(v)} samples')

        # Workflow 1: mistakenness (only if OOF available from a previous run)
        prev_oof_csv = SAVE / 'labels_corrected_v25_b4.csv'
        if prev_oof_csv.exists():
            print('Prior OOF found — running compute_mistakenness')
            # Implementation: attach predictions to samples, then fob.compute_mistakenness
            # (Skipped here for brevity — runs after Phase 1 in this notebook)

        session = fo.launch_app(dataset, port=5151)
        print('FiftyOne app at http://localhost:5151')
    except ImportError:
        print('FiftyOne not installed — skip')
    except Exception as e:
        print(f'FiftyOne error: {e}')
else:
    print('USE_FIFTYONE=False — skip. On Kaggle this is the right default.')
    print('Cleanlab triage (cell 10) provides the same label-noise signal '
          'without needing a browser.')


In [ ]:
# =========================================================
# CELL 7: 🔴 PRIORITY 1 — Rule-based label correction
# =========================================================
# Host confirmed (forum reply to Allen Joseph Antony):
#   "Areas of annotations with the same class/label must be aggregated."
# 4th-place team: per-class area summing reconstructs 100% of train labels.
#
# v23 bug fixes both included here:
#   FIX 1: _get_ann_area() with shoelace polygon fallback (no silent 0-areas)
#   FIX 2: img_cat_exists for always-faulty (existence check, not area>0)

img_id_to_stem = {img['id']: Path(img['file_name']).stem for img in train_ann_data['images']}

def _shoelace(coords):
    pts = [(coords[i], coords[i+1]) for i in range(0, len(coords)-1, 2)]
    if len(pts) < 3: return 0.0
    n = len(pts)
    a = sum(pts[i][0]*pts[(i+1)%n][1] - pts[(i+1)%n][0]*pts[i][1] for i in range(n))
    return abs(a) / 2.0

def _get_ann_area(ann):
    """Polygon area: ann['area'] if set, else shoelace, else bbox w*h."""
    area = float(ann.get('area', 0))
    if area > 0: return area
    seg = ann.get('segmentation', [])
    if seg:
        return sum(_shoelace(p) for p in seg if isinstance(p, list) and len(p) >= 6)
    bbox = ann.get('bbox', [])
    if len(bbox) == 4: return float(bbox[2] * bbox[3])
    return 0.0

# Aggregate
img_cat_areas  = defaultdict(lambda: defaultdict(float))
img_cat_exists = defaultdict(set)
for ann in train_ann_data['annotations']:
    cn = cat_id_to_name.get(ann['category_id'], f'unknown_{ann["category_id"]}')
    st = img_id_to_stem.get(ann['image_id'])
    if st is None: continue
    img_cat_areas[st][cn]  += _get_ann_area(ann)
    img_cat_exists[st].add(cn)

print(f'Images with annotations: {len(img_cat_areas)}')

def derive_label_rule_based(stem):
    existing = img_cat_exists.get(stem, set())
    cats     = img_cat_areas.get(stem, {})
    if not cats and not existing:
        return 0, 'no_annotations'
    # Rule 1: always-faulty → existence
    for cn in existing:
        if cn in ALWAYS_FAULTY_CATS:
            return 1, f'always_faulty:{cn}'
    # Rule 2: conditional → summed area ≥ threshold
    for cn, area in cats.items():
        if cn in CONDITIONAL_THRESHOLDS and area >= CONDITIONAL_THRESHOLDS[cn]:
            return 1, f'conditional:{cn}(area={area:.0f})'
    return 0, 'clean'

labels_dict_orig = dict(zip(labels_orig['stem'], labels_orig['target']))
corrections = []
unknown_cats = set()
for stem, csv_target in labels_dict_orig.items():
    for cn in img_cat_areas.get(stem, {}):
        if (cn not in ALWAYS_FAULTY_CATS and cn not in CONDITIONAL_THRESHOLDS
            and cn not in ALWAYS_GOOD_CATS and cn.lower() not in {'roi', 'background'}):
            unknown_cats.add(cn)
    rule_label, reason = derive_label_rule_based(stem)
    if rule_label != csv_target:
        corrections.append({'image_id': stem, 'csv_label': int(csv_target),
                            'rule_label': int(rule_label), 'reason': reason})

print(f'Mismatches found: {len(corrections)}')
if unknown_cats:
    print(f'⚠️ Unknown cats: {unknown_cats}')
for c in corrections[:8]:
    print(f"  {c['image_id']}: {c['csv_label']} → {c['rule_label']}  ({c['reason']})")

labels_corrected = labels_orig.copy()
if LABEL_CORRECTION_ENABLED and corrections:
    cmap = {c['image_id']: c['rule_label'] for c in corrections}
    def _maybe(row):
        return cmap.get(Path(str(row['image_id'])).stem, row['target'])
    labels_corrected['target'] = labels_corrected.apply(_maybe, axis=1)
    flipped = (labels_corrected.target != labels_orig.target).sum()
    print(f'✅ Applied {flipped} corrections')

labels_corrected.to_csv(SAVE / f'labels_corrected_{VERSION_TAG}.csv', index=False)
pd.DataFrame(corrections).to_csv(SAVE / f'priority1_audit_{VERSION_TAG}.csv', index=False)

y_corrected = labels_corrected['target'].values.astype(np.int64)
print(f'class1 ratio after correction: {y_corrected.mean():.4f}')


In [ ]:
# =========================================================
# CELL 8: Attach bottle_type to labels_corrected
# =========================================================
# btype_map already built in cell 4. Now attach to labels_corrected as a column.

labels_corrected['stem']     = labels_corrected['image_id'].apply(lambda x: Path(str(x)).stem)
labels_corrected['btype_id'] = labels_corrected['stem'].map(btype_map).fillna(-1).astype(int)

missing = (labels_corrected.btype_id == -1).sum()
if missing > 0:
    print(f'⚠️ {missing} training images missing from bottletypes.csv')
    sample_missing = labels_corrected[labels_corrected.btype_id == -1].head()
    print(sample_missing[['image_id']])
else:
    print(f'✅ All {len(labels_corrected)} train images have a bottle type assigned')

print('\nPer-type class-1 rate (corrected labels):')
for tname, tid in BOTTLE_TYPE_CANONICAL.items():
    sub = labels_corrected[labels_corrected.btype_id == tid]
    if len(sub):
        print(f'  {tname:7s}: n={len(sub):>5d}  class1={sub.target.mean():.4f}  '
              f'(faulty={int(sub.target.sum())})')


In [ ]:
# =========================================================
# CELL 9: LEADERBOARD SPLIT - use all labeled data for CV
# =========================================================
# Leaderboard mode disables the extra 10% holdout so every labeled image can
# contribute to the 5-fold CV ensemble. OOF predictions from the folds still
# provide threshold/weight tuning, but there is no separate untouched guardrail.

train_idx_all = np.arange(len(labels_corrected))
holdout_idx = np.array([], dtype=int)

_hold_p  = SAVE / f'holdout_indices_{VERSION_TAG}.npy'
_train_p = SAVE / f'train_indices_{VERSION_TAG}.npy'
np.save(_hold_p, holdout_idx)
np.save(_train_p, train_idx_all)

labels_train_active = labels_corrected.reset_index(drop=True).copy()
labels_holdout = labels_corrected.iloc[[]].reset_index(drop=True).copy()
y_holdout = np.array([], dtype=np.int64)

labels_train_active.to_csv(SAVE / f'labels_train_active_{VERSION_TAG}.csv', index=False)
labels_holdout.to_csv(SAVE / f'labels_holdout_{VERSION_TAG}.csv', index=False)

print('Leaderboard mode: using all labeled data for 5-fold training.')
print(f'  Train active: {len(labels_train_active)} (100%)')
print(f'  Holdout:      {len(labels_holdout)} (0%)')
print(f'  Train class1 ratio: {labels_train_active.target.mean():.4f}')

print('\nPer-type training coverage:')
for tname, tid in BOTTLE_TYPE_CANONICAL.items():
    t_n = (labels_train_active.btype_id == tid).sum()
    print(f'  {tname:7s}: train={t_n:>5d}')

print('\nOOF fold validation replaces the separate holdout for leaderboard mode.')


In [ ]:
# =========================================================
# CELL 10: Cleanlab two-pass triage + tri-bucket A/B/C
# =========================================================
# First pass: uses OOF probs from any previous v24/v25 run if available;
#             otherwise defaults to all-bucket-A (safe fallback).
# Second pass: re-run this cell AFTER Phase 1 completes for informed triage.
#
# Buckets:
#   A — clean, weight = 1.0
#   B — ambiguous (model 0.70-0.90 confident against label), weight = 0.3, NLS α = -0.1
#   C — likely error (model >0.90 confident against label), weight = 0.0 (drop)

CLEANLAB_AVAILABLE = False
if USE_CLEANLAB:
    try:
        from cleanlab.filter import find_label_issues
        CLEANLAB_AVAILABLE = True
    except ImportError:
        print('cleanlab not installed — falling back to bucket-A for all')

def load_best_oof_for_triage(tag):
    """Build OOF prob array aligned to labels_train_active indices."""
    arr = np.full(len(labels_train_active), np.nan, dtype=np.float32)
    found = False
    for fold in range(N_FOLDS):
        p = SAVE / f'oof_{tag}_p1_fold{fold}.npy'
        i = SAVE / f'oof_idx_{tag}_p1_fold{fold}.npy'
        if p.exists() and i.exists():
            idx_in_active = np.load(i); probs = np.load(p)
            for local_idx, prob in zip(idx_in_active, probs):
                if 0 <= local_idx < len(arr):
                    arr[local_idx] = prob
            found = True
    return arr, found

oof_for_triage, oof_found = load_best_oof_for_triage(VERSION_TAG)

# Optionally use prior version's OOF on first run (v25/v24/v23)
if not oof_found:
    # Build stem→active_idx lookup once (faster than per-sample linear search)
    stem_to_active = {s: i for i, s in enumerate(labels_train_active['stem'].values)}
    for prev in ['v25_b4', 'v24_b4', 'v23_b4']:
        prev_oof = np.full(len(labels_train_active), np.nan, dtype=np.float32)
        prev_found = False
        for fold in range(N_FOLDS):
            for root, _, files in os.walk(PROJECT_ROOT / 'outputs'):
                fn = f'oof_{prev}_pl_fold{fold}.npy'
                idx_fn = f'oof_idx_{prev}_pl_fold{fold}.npy'
                if fn in files and idx_fn in files:
                    p = Path(root)/fn; i = Path(root)/idx_fn
                    indices = np.load(i); probs = np.load(p)
                    # Prior indices were against full labels_corrected (90% +
                    # 10% holdout). Map via stem to current active set.
                    for ci, pr in zip(indices, probs):
                        if 0 <= ci < len(labels_corrected):
                            stem = labels_corrected.iloc[ci]['stem']
                            ai = stem_to_active.get(stem)
                            if ai is not None: prev_oof[ai] = pr
                    prev_found = True
                    break
            if prev_found: break
        if prev_found:
            oof_for_triage = prev_oof
            print(f'Using OOF from {prev} for initial triage')
            break

valid_mask = ~np.isnan(oof_for_triage)
n_valid = int(valid_mask.sum())
print(f'OOF coverage for cleanlab: {n_valid}/{len(labels_train_active)}')

sample_weights = np.ones(len(labels_train_active), dtype=np.float32)
nls_flags      = np.zeros(len(labels_train_active), dtype=bool)
bucket_labels  = np.array(['A'] * len(labels_train_active), dtype='U1')

y_train_active = labels_train_active['target'].values.astype(np.int64)

if n_valid > 100 and CLEANLAB_AVAILABLE:
    p1 = np.clip(oof_for_triage, 1e-6, 1-1e-6)
    p_oof_matrix = np.column_stack([1-p1, p1])
    valid_idx = np.where(valid_mask)[0]
    issues_local = find_label_issues(
        labels=y_train_active[valid_idx],
        pred_probs=p_oof_matrix[valid_idx],
        return_indices_ranked_by='self_confidence'
    )
    global_issue_idx = valid_idx[issues_local]
    print(f'Cleanlab flagged {len(global_issue_idx)} likely label issues')

    for gi in global_issue_idx:
        prob = float(oof_for_triage[gi])
        label = int(y_train_active[gi])
        conf_against = prob if label == 0 else (1 - prob)
        if conf_against >= BUCKET_C_DROP_CONF:
            bucket_labels[gi] = 'C'
            sample_weights[gi] = 0.0
        elif conf_against >= BUCKET_B_FLAG_CONF:
            bucket_labels[gi] = 'B'
            sample_weights[gi] = BUCKET_B_WEIGHT
            nls_flags[gi] = True

    nA = (bucket_labels == 'A').sum()
    nB = (bucket_labels == 'B').sum()
    nC = (bucket_labels == 'C').sum()
    print(f'\nTri-bucket:')
    print(f'  A (clean, w=1.0):           {nA:>5d}')
    print(f'  B (ambiguous, w={BUCKET_B_WEIGHT}, NLS): {nB:>5d}')
    print(f'  C (drop, w=0.0):            {nC:>5d}')
else:
    if not CLEANLAB_AVAILABLE:
        print('cleanlab unavailable — all samples bucket A')
    else:
        print(f'Insufficient OOF ({n_valid} samples) — all bucket A on first pass.')
        print('After Phase 1 completes, re-run this cell for informed triage.')

# Attach to labels_train_active
labels_train_active['sample_weight'] = sample_weights
labels_train_active['nls']           = nls_flags
labels_train_active['bucket']        = bucket_labels
labels_train_active.to_csv(SAVE / f'triage_{VERSION_TAG}.csv', index=False)


In [ ]:
# =========================================================
# CELL 11: Transforms (no Gabor — MaxViT uses RGB only)
# =========================================================
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

# Gabor disabled for MaxViT — keep empty kernel list for dataset compatibility
GABOR_KERNELS = []
print('Gabor disabled (MaxViT RGB-only)')

def build_transforms(size=IMG_SIZE):
    train_tf = A.Compose([
        A.Resize(size, size),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.10, rotate_limit=8,
                           border_mode=cv2.BORDER_REFLECT_101, p=0.7),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=15, p=1.0),
            A.CLAHE(clip_limit=2.0, p=1.0),
            A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=1.0),
        ], p=0.7),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.MotionBlur(blur_limit=5, p=1.0),
            A.GaussNoise(var_limit=(10, 50), p=1.0),
        ], p=0.3),
        A.CoarseDropout(max_holes=4, max_height=int(size*0.1), max_width=int(size*0.1),
                        min_holes=1, fill_value=0, p=0.3),
        A.Normalize(mean=NORM_MEAN, std=NORM_STD),
        ToTensorV2(),
    ])
    # Strong aug 2nd view (kept for API parity; consistency not used here)
    strong_tf = train_tf
    valid_tf = A.Compose([
        A.Resize(size, size),
        A.Normalize(mean=NORM_MEAN, std=NORM_STD),
        ToTensorV2(),
    ])
    return train_tf, strong_tf, valid_tf

train_tf, strong_tf, valid_tf = build_transforms(IMG_SIZE)

def tta_transform(mode, size=IMG_SIZE):
    base = [A.Resize(size, size)]
    if mode == 'hflip':   base.append(A.HorizontalFlip(p=1.0))
    elif mode == 'rot5':  base.append(A.Rotate(limit=(5, 5), border_mode=cv2.BORDER_REFLECT_101, p=1.0))
    elif mode == 'rot-5': base.append(A.Rotate(limit=(-5, -5), border_mode=cv2.BORDER_REFLECT_101, p=1.0))
    elif mode == 'scale09':
        s = int(size * 0.9)
        base = [A.Resize(s, s), A.PadIfNeeded(size, size, border_mode=cv2.BORDER_REFLECT_101)]
    elif mode == 'scale11':
        s = int(size * 1.1)
        base = [A.Resize(s, s), A.CenterCrop(size, size)]
    base += [A.Normalize(mean=NORM_MEAN, std=NORM_STD), ToTensorV2()]
    return A.Compose(base)

tta_tf = {m: tta_transform(m, IMG_SIZE) for m in TTA_MODES}
print('Transforms built. TTA modes:', TTA_MODES)


In [ ]:
# =========================================================
# CELL 12: Classes — BottleDataset, MaxViTClassifier, Loss, EMA, Scheduler
# =========================================================
# MaxViT version: NO GAIN hook, NO Gabor branch. Dataset still returns the
# 8-tuple so train_utils stays compatible (gabor_t / gain_mask are dummy zeros).
def crop_roi(img, bbox, padding=8):
    if bbox is None: return img
    x, y, w, h = bbox; ih, iw = img.shape[:2]
    x1 = max(0, int(x)-padding); y1 = max(0, int(y)-padding)
    x2 = min(iw, int(x+w)+padding); y2 = min(ih, int(y+h)+padding)
    return img[y1:y2, x1:x2] if (x2>x1 and y2>y1) else img

_MASK_CACHE = None  # GAIN disabled


# ── BottleDataset (returns 8-tuple for train_util compatibility) ──
class BottleDataset(Dataset):
    """Returns 8-tuple: (img_t, gabor_t, img_t2, gain_mask, label, weight, nls_flag, btype)
    For MaxViT, gabor_t and gain_mask are always dummy zeros (no Gabor/GAIN).
    alt_dirs: fallback search directories for pseudo/TEST images.
    """
    def __init__(self, df, img_dir, roi_map, transform, ann_cache=None,
                 is_test=False, use_gabor=USE_GABOR_STREAM, use_gain=USE_GAIN,
                 second_view_tf=None, alt_dirs=None, alt_roi_maps=None):
        self.df         = df.reset_index(drop=True)
        self.img_dir    = Path(img_dir)
        self.roi_map    = roi_map
        self.tf         = transform
        self.tf2        = second_view_tf
        self.cache      = ann_cache
        self.is_test    = is_test
        self.alt_dirs   = [Path(d) for d in (alt_dirs or [])]
        self.alt_roi_maps = alt_roi_maps or []

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        iid  = str(row['image_id'])
        stem = Path(iid).stem

        img = None
        roi_bbox = self.roi_map.get(stem)
        for d_idx, d in enumerate([self.img_dir] + self.alt_dirs):
            for ext in ['', '.png', '.jpg', '.jpeg']:
                p = d / (iid if ext == '' else f'{stem}{ext}')
                if p.exists():
                    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
                    if d_idx > 0 and d_idx <= len(self.alt_roi_maps):
                        roi_bbox = self.alt_roi_maps[d_idx-1].get(stem, roi_bbox)
                    break
            if img is not None: break
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

        img_cropped = crop_roi(img, roi_bbox)
        img_resized = cv2.resize(img_cropped, (IMG_SIZE, IMG_SIZE))

        # Dummy gabor (MaxViT doesn't use it)
        gabor_t = torch.zeros(GABOR_OUT_CHANNELS, IMG_SIZE//4, IMG_SIZE//4)

        img_t = self.tf(image=img_resized)['image']
        img_t2 = self.tf2(image=img_resized)['image'] if self.tf2 is not None \
                 else torch.zeros_like(img_t)

        # Dummy gain mask (MaxViT doesn't use it)
        gain_mask = torch.zeros(GAIN_FEAT_SIZE, GAIN_FEAT_SIZE)

        btype = int(btype_map.get(stem, -1))

        if self.is_test:
            return img_t, gabor_t, img_t2, gain_mask, \
                   torch.tensor(-1.0), torch.tensor(1.0), torch.tensor(False), btype

        target = float(row.get('target', 0))
        weight = float(row.get('sample_weight', 1.0))
        nls    = bool(row.get('nls', False))
        return (img_t, gabor_t, img_t2, gain_mask,
                torch.tensor(target, dtype=torch.float32),
                torch.tensor(weight, dtype=torch.float32),
                torch.tensor(nls, dtype=torch.bool),
                btype)


# ── MaxViTClassifier ──────────────────────────────────────────
class BottleClsMT(nn.Module):
    """MaxViT-Tiny backbone + binary head + aux bottle-type head.
    Named BottleClsMT so train_utils / pl-gen code stays identical to B4.
    NO GAIN hook, NO Gabor branch. LayerNorm-based (no BatchNorm in backbone).
    """
    def __init__(self, name=MODEL_NAME, n_btypes=N_BOTTLE_TYPES,
                 dropout=DROPOUT, drop_path_rate=DROP_PATH_RATE_P1,
                 use_gabor=False, pretrained=True,
                 use_grad_checkpoint=USE_GRAD_CHECKPOINT):
        super().__init__()
        self.use_gabor = False
        self.backbone = timm.create_model(
            name, pretrained=pretrained, num_classes=0,
            drop_rate=dropout, drop_path_rate=drop_path_rate)
        f = self.backbone.num_features

        if use_grad_checkpoint and hasattr(self.backbone, 'set_grad_checkpointing'):
            self.backbone.set_grad_checkpointing(enable=True)

        self.gabor_head = None
        cls_in = f

        self.cls_head = nn.Sequential(
            nn.Dropout(dropout * 0.5),
            nn.Linear(cls_in, 256),
            nn.LayerNorm(256),          # LayerNorm (matches MaxViT, BN-free)
            nn.GELU(),
            nn.Dropout(dropout * 0.3),
            nn.Linear(256, 1),
        )
        self.btype_head = nn.Sequential(
            nn.Dropout(dropout * 0.3),
            nn.Linear(f, n_btypes),
        )
        self._feat_map = None  # no GAIN hook

    def forward(self, x, gabor_x=None, return_btype=False):
        feat = self.backbone(x)
        cls = self.cls_head(feat).view(-1)
        if return_btype:
            return cls, self.btype_head(feat)
        return cls

    def get_feat_map(self):
        return self._feat_map  # always None for MaxViT


# ── Loss, EMA, Scheduler, Temperature (identical to B4) ──────
class ComboLossNLS(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, w_bce=0.5, w_focal=0.5):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
        self.w_bce = w_bce; self.w_focal = w_focal
    def forward(self, logits, targets, weights=None, nls_mask=None):
        if nls_mask is not None and nls_mask.any():
            targets = targets.clone()
            targets[nls_mask] = targets[nls_mask] + NLS_ALPHA * (0.5 - targets[nls_mask])
        bce   = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p     = torch.sigmoid(logits)
        pt    = p * targets + (1 - p) * (1 - targets)
        focal = self.alpha * ((1 - pt) ** self.gamma) * bce
        loss  = self.w_bce * bce + self.w_focal * focal
        if weights is not None:
            loss = loss * weights
        return loss.mean()


class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.clone().detach() for n, p in model.named_parameters() if p.requires_grad}
    def update(self, model):
        for n, p in model.named_parameters():
            if not p.requires_grad: continue
            self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)
    def apply_to(self, model):
        for n, p in model.named_parameters():
            if n in self.shadow: p.data.copy_(self.shadow[n])


class WarmupCosineScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, base_lr, min_lr=1e-7):
        self.opt = optimizer; self.warmup = warmup_steps; self.total = total_steps
        self.base = base_lr; self.min_lr = min_lr; self.step_n = 0
    def step(self):
        self.step_n += 1
        if self.step_n <= self.warmup:
            lr = self.base * self.step_n / max(1, self.warmup)
        else:
            t  = (self.step_n - self.warmup) / max(1, self.total - self.warmup)
            lr = self.min_lr + 0.5 * (self.base - self.min_lr) * (1 + math.cos(math.pi * t))
        for g in self.opt.param_groups: g['lr'] = lr


class TemperatureScaling(nn.Module):
    def __init__(self):
        super().__init__()
        self.T = nn.Parameter(torch.ones(1) * 1.0)
    def forward(self, logits): return logits / self.T

def fit_temperature(logits, labels, max_iter=100):
    ts = TemperatureScaling().to(DEVICE)
    opt = torch.optim.LBFGS([ts.T], lr=0.01, max_iter=max_iter)
    lt = torch.tensor(logits, dtype=torch.float32, device=DEVICE)
    yt = torch.tensor(labels, dtype=torch.float32, device=DEVICE)
    def closure():
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(ts(lt), yt)
        loss.backward(); return loss
    opt.step(closure)
    return float(ts.T.detach().cpu())

print('Classes defined: BottleDataset (8-tuple, dummy gabor/gain),')
print('  BottleClsMT = MaxViT-Tiny + cls_head (LayerNorm) + btype_head, NO GAIN/Gabor.')
print('  ComboLossNLS, EMA, WarmupCosineScheduler, TemperatureScaling.')


In [ ]:
# =========================================================
# CELL 13: SWA + recal_bn (on CLEAN LABELED SUBSET ONLY)
# =========================================================
# v22 lesson: recal_bn on pseudo-augmented set destroys F1.
# v26 fix: recal_bn always uses the clean labeled training subset only.

def average_state_dicts(states):
    avg = copy.deepcopy(states[0])
    for k in avg.keys():
        if not torch.is_floating_point(avg[k]):
            continue
        target_device = avg[k].device
        target_dtype = avg[k].dtype
        acc = torch.zeros_like(avg[k].detach().cpu(), dtype=torch.float64)
        for s in states:
            acc += s[k].detach().cpu().double()
        avg[k] = (acc / len(states)).to(dtype=target_dtype, device=target_device)
    return avg

@torch.no_grad()
def recal_bn_on_clean(model, clean_loader, device=DEVICE, max_batches=None):
    """Recalibrate BN stats using ONLY the clean labeled subset (never pseudo)."""
    model.train()
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            m.reset_running_stats(); m.momentum = 0.1
    for i, batch in enumerate(clean_loader):
        imgs = batch[0].to(device, non_blocking=True)
        gabor = batch[1].to(device, non_blocking=True) if USE_GABOR_STREAM else None
        model(imgs, gabor_x=gabor)
        if max_batches and i + 1 >= max_batches: break
    model.eval()

print('SWA + recal_bn_on_clean helpers defined.')


In [ ]:
# =========================================================
# CELL 14: Train utilities — GAIN, consistency, train_epoch (grad accum),
#          eval_fold, predict_test_tta, predict_mc_dropout
# =========================================================

def compute_gain_loss(model, imgs, gain_masks, gabor_x=None):
    """GAIN L_e: MSE between Grad-CAM and polygon mask. Min 2 faulty for BN."""
    faulty_idx = (gain_masks.sum(dim=[-2, -1]) > 0).nonzero(as_tuple=True)[0]
    if len(faulty_idx) < 2:
        return torch.tensor(0.0, device=imgs.device)

    imgs_f  = imgs[faulty_idx]
    masks_f = gain_masks[faulty_idx]
    gab_f   = gabor_x[faulty_idx] if gabor_x is not None else None

    logits_f = model(imgs_f, gabor_x=gab_f)
    feat = model.get_feat_map()
    if feat is None:
        return torch.tensor(0.0, device=imgs.device)
    feat_f = feat[faulty_idx] if feat.shape[0] == imgs.shape[0] else feat

    score = logits_f.sum()
    grads = torch.autograd.grad(score, feat_f, create_graph=True,
                                retain_graph=True, allow_unused=True)[0]
    if grads is None:
        return torch.tensor(0.0, device=imgs.device)

    weights = grads.mean(dim=[-2, -1], keepdim=True)
    cam = F.relu((weights * feat_f).sum(dim=1))
    cmin = cam.flatten(1).min(dim=1)[0].view(-1, 1, 1)
    cmax = cam.flatten(1).max(dim=1)[0].view(-1, 1, 1)
    cam_n = (cam - cmin) / (cmax - cmin + 1e-8)

    h, w = cam_n.shape[-2:]
    if masks_f.shape[-2:] != (h, w):
        masks_f = F.interpolate(masks_f.unsqueeze(1).float(), size=(h, w),
                                mode='bilinear', align_corners=False).squeeze(1)
    return ((cam_n - masks_f) ** 2).mean()


def consistency_loss_kl(logits_v1, logits_v2):
    p1 = torch.sigmoid(logits_v1).clamp(1e-7, 1-1e-7)
    p2 = torch.sigmoid(logits_v2).clamp(1e-7, 1-1e-7)
    kl = p1*(torch.log(p1)-torch.log(p2)) + (1-p1)*(torch.log(1-p1)-torch.log(1-p2))
    return kl.mean()


def mixup_cutmix_for_batch(imgs, labels, weights, alpha=0.2, p_cutmix=0.5, apply_p=0.5):
    if random.random() > apply_p:
        return imgs, labels, labels, 1.0
    a_mask = weights >= 1.0
    if a_mask.sum() < 2:
        return imgs, labels, labels, 1.0
    idx_a = torch.where(a_mask)[0]
    perm  = idx_a[torch.randperm(len(idx_a), device=imgs.device)]
    lam   = float(np.random.beta(alpha, alpha))
    if random.random() < p_cutmix:
        H, W = imgs.shape[-2:]
        rh = int(H * math.sqrt(1 - lam)); rw = int(W * math.sqrt(1 - lam))
        cy = np.random.randint(H); cx = np.random.randint(W)
        y1 = max(0, cy - rh//2); y2 = min(H, cy + rh//2)
        x1 = max(0, cx - rw//2); x2 = min(W, cx + rw//2)
        imgs[idx_a, :, y1:y2, x1:x2] = imgs[perm, :, y1:y2, x1:x2]
        lam = 1 - ((y2-y1)*(x2-x1)/(H*W))
    else:
        imgs[idx_a] = lam * imgs[idx_a] + (1-lam) * imgs[perm]
    labels_b = labels.clone(); labels_b[idx_a] = labels[perm]
    return imgs, labels, labels_b, lam


def train_epoch(model, loader, opt, sched, scaler, ema, criterion,
                use_mix=True, btype_weight=0.10, gain_omega=GAIN_OMEGA,
                consistency_w=0.0, accum_steps=GRAD_ACCUM_STEPS):
    model.train()
    losses, gain_losses, cons_losses = [], [], []
    pbar = tqdm(loader, desc='train', leave=False)
    opt.zero_grad(set_to_none=True)

    for step_i, batch in enumerate(pbar):
        imgs, gabor_imgs, imgs_v2, gain_masks, labels, weights, nls_flags, btypes = batch
        imgs       = imgs.to(DEVICE, non_blocking=True)
        gabor_imgs = gabor_imgs.to(DEVICE, non_blocking=True) if USE_GABOR_STREAM else None
        imgs_v2    = imgs_v2.to(DEVICE, non_blocking=True)
        gain_masks = gain_masks.to(DEVICE, non_blocking=True)
        labels     = labels.to(DEVICE, non_blocking=True).float()
        weights    = weights.to(DEVICE, non_blocking=True).float()
        nls_flags  = nls_flags.to(DEVICE, non_blocking=True)
        btypes     = btypes.to(DEVICE, non_blocking=True).long()

        if use_mix:
            imgs, ya, yb, lam = mixup_cutmix_for_batch(imgs, labels, weights)
        else:
            ya, yb, lam = labels, labels, 1.0

        with amp_autocast():
            cls_logits, btype_logits = model(imgs, gabor_x=gabor_imgs, return_btype=True)
            if use_mix:
                la = criterion(cls_logits, ya, weights=weights, nls_mask=nls_flags)
                lb = criterion(cls_logits, yb, weights=weights, nls_mask=nls_flags)
                loss_cls = lam * la + (1-lam) * lb
            else:
                loss_cls = criterion(cls_logits, labels, weights=weights, nls_mask=nls_flags)

            valid_bt = btypes >= 0
            if valid_bt.any():
                loss = loss_cls + btype_weight * F.cross_entropy(btype_logits[valid_bt], btypes[valid_bt])
            else:
                loss = loss_cls

            if consistency_w > 0 and imgs_v2.abs().sum() > 0:
                logits_v2 = model(imgs_v2, gabor_x=gabor_imgs)
                cons = consistency_loss_kl(cls_logits, logits_v2)
                loss = loss + consistency_w * cons
                cons_losses.append(float(cons.detach()))

            loss = loss / accum_steps  # scale for accumulation

        # GAIN: outside autocast, disabled during accumulation sub-steps to save memory
        if USE_GAIN and gain_omega > 0 and (step_i % accum_steps == 0):
            try:
                with amp_autocast(enabled=False):
                    l_e = compute_gain_loss(model, imgs.float(), gain_masks.float(),
                                            gabor_x=gabor_imgs.float() if gabor_imgs is not None else None)
                if l_e.item() > 0:
                    (gain_omega * l_e / accum_steps).backward()
                    gain_losses.append(float(l_e.detach()))
            except Exception:
                pass

        scaler.scale(loss).backward()

        if (step_i + 1) % accum_steps == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update()
            sched.step()
            if ema is not None: ema.update(model)
            opt.zero_grad(set_to_none=True)

        losses.append(float(loss.detach()) * accum_steps)
        pbar.set_postfix(loss=f'{np.mean(losses[-50:]):.4f}',
                         gain=f'{np.mean(gain_losses) if gain_losses else 0:.3f}',
                         cons=f'{np.mean(cons_losses) if cons_losses else 0:.3f}')
    return float(np.mean(losses))


@torch.no_grad()
def eval_fold(model, loader):
    model.eval()
    probs, ys, bts, wts = [], [], [], []
    for batch in loader:
        imgs, gabor_imgs, _, _, labels, weights, _, btypes = batch
        imgs = imgs.to(DEVICE, non_blocking=True)
        gabor_imgs = gabor_imgs.to(DEVICE, non_blocking=True) if USE_GABOR_STREAM else None
        with amp_autocast():
            logits = model(imgs, gabor_x=gabor_imgs)
        probs.append(torch.sigmoid(logits).float().cpu().numpy())
        ys.append(labels.numpy()); bts.append(btypes.numpy()); wts.append(weights.numpy())
    return np.concatenate(probs), np.concatenate(ys), np.concatenate(bts), np.concatenate(wts)


@torch.no_grad()
def predict_test_tta(model, df_test, img_dir, roi_map, tta_modes=TTA_MODES, bs=BATCH_SIZE):
    model.eval()
    all_probs, bts_out = None, None
    for mode in tta_modes:
        ds = BottleDataset(df_test, img_dir, roi_map, tta_tf[mode], is_test=True)
        ld = DataLoader(ds, batch_size=bs, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        ps, bts = [], []
        for batch in ld:
            imgs, gabor_imgs, _, _, _, _, _, btypes = batch
            imgs = imgs.to(DEVICE, non_blocking=True)
            gabor_imgs = gabor_imgs.to(DEVICE, non_blocking=True) if USE_GABOR_STREAM else None
            with amp_autocast():
                logits = model(imgs, gabor_x=gabor_imgs)
            ps.append(torch.sigmoid(logits).float().cpu().numpy())
            bts.append(btypes.numpy())
        ps = np.concatenate(ps)
        all_probs = ps if all_probs is None else all_probs + ps
        bts_out = np.concatenate(bts)
    return all_probs / len(tta_modes), bts_out


@torch.no_grad()
def predict_mc_dropout(model, df_test, img_dir, roi_map, T=MC_T, bs=16):
    model.train()  # dropout active
    ds = BottleDataset(df_test, img_dir, roi_map, valid_tf, is_test=True)
    ld = DataLoader(ds, batch_size=bs, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    runs = []
    for t in range(T):
        run_p = []
        for batch in ld:
            imgs, gabor_imgs, _, _, _, _, _, _ = batch
            imgs = imgs.to(DEVICE, non_blocking=True)
            gabor_imgs = gabor_imgs.to(DEVICE, non_blocking=True) if USE_GABOR_STREAM else None
            with amp_autocast():
                logits = model(imgs, gabor_x=gabor_imgs)
            run_p.append(torch.sigmoid(logits).float().cpu().numpy())
        runs.append(np.concatenate(run_p))
    stacked = np.stack(runs, axis=0)
    p_mean  = stacked.mean(0); p_var = stacked.var(0)
    eps = 1e-9
    entropy = -(p_mean*np.log(p_mean+eps) + (1-p_mean)*np.log(1-p_mean+eps))
    return p_mean, p_var, entropy


def best_threshold(y, p, lo=0.20, hi=0.81, step=0.005):
    bt, bf = 0.5, 0.0
    for t in np.arange(lo, hi, step):
        f = f1_score(y, (p >= t).astype(int), zero_division=0)
        if f > bf: bf, bt = f, t
    return bt, bf


def fold_completed(phase, fold):
    return (SAVE / f'model_{VERSION_TAG}_{phase}_fold{fold}.pth').exists()

print('Train utilities defined.')
print(f'  Gradient accumulation: {GRAD_ACCUM_STEPS} steps (effective BS = {BATCH_SIZE*GRAD_ACCUM_STEPS})')


In [ ]:
# =========================================================
# CELL 15: Joint-stratified 5-fold splits (btype × target)
# =========================================================
# Folds operate on labels_train_active (90% portion). Holdout is never in any fold.

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
strat_key_active = (labels_train_active['btype_id'].clip(0).astype(int) * 2
                    + labels_train_active['target'].astype(int)).values
SKF_SPLITS = list(skf.split(np.zeros(len(labels_train_active)), strat_key_active))

print('Joint-stratified folds (over train_active only — holdout is OUT):')
print(f'{"fold":>4s}  {"train":>6s}  {"val":>5s}  {"V+":>3s} {"V-":>3s}  '
      f'{"E+":>3s} {"E-":>3s}  {"N+":>4s} {"N-":>4s}  c1%')
for fold, (tr, va) in enumerate(SKF_SPLITS):
    sub = labels_train_active.iloc[va]
    cnt = {}
    for tname, tid in BOTTLE_TYPE_CANONICAL.items():
        s = sub[sub.btype_id == tid]
        cnt[f'{tname}+'] = int(s.target.sum())
        cnt[f'{tname}-'] = int(len(s) - s.target.sum())
    print(f'{fold:>4d}  {len(tr):>6d}  {len(va):>5d}  '
          f'{cnt["VICHY+"]:>3d} {cnt["VICHY-"]:>3d}  '
          f'{cnt["EURO+"]:>3d} {cnt["EURO-"]:>3d}  '
          f'{cnt["NRW+"]:>4d} {cnt["NRW-"]:>4d}  '
          f'{sub.target.mean():.4f}')

# Checkpoint status
print('\nCheckpoint status:')
for phase in ['p1', 'p2', 'pl']:
    done = sum(fold_completed(phase, f) for f in range(N_FOLDS))
    print(f'  {phase}: {done}/{N_FOLDS} folds completed')


In [ ]:
# =========================================================
# CELL 16: train_fold() — curriculum + GAIN + NLS + consistency
# =========================================================
TRAIN_IMG = DATA / 'train_images'
TEST_IMG  = DATA / 'test_images'


def _stem_sets_from_labels(df):
    """Compute (clean_A, active_AB) stem sets from a labels df with 'bucket' col."""
    if 'bucket' not in df.columns:
        all_stems = set(df['stem']) if 'stem' in df.columns \
                    else set(df['image_id'].apply(lambda x: Path(str(x)).stem))
        return all_stems, all_stems
    return (set(df.loc[df.bucket == 'A', 'stem']),
            set(df.loc[df.bucket != 'C', 'stem']))


def train_fold(phase, fold, df_train, df_val,
               epochs, lr,
               init_state=None,
               use_swa=True,
               use_recal_bn=True,
               recal_loader=None,
               use_mix=True,
               pseudo_df=None,
               curriculum='none',
               drop_path_rate=DROP_PATH_RATE_P1,
               use_consistency=False,
               anneal_gain=True):
    """
    curriculum: 'none' | 'bucket_a_only' | 'bucket_ab'
    use_recal_bn + recal_loader: BN recal on the provided CLEAN labeled loader (never pseudo).
    use_consistency: 2nd augmented view + KL loss (PL phase).
    """
    print(f'\n{"="*60}\n[{phase.upper()} | curr={curriculum} | dp={drop_path_rate} | '
          f'cons={use_consistency}] Fold {fold}\n{"="*60}')

    # Recompute stem sets from current labels_train_active (picks up re-run cleanlab)
    clean_stems_A, active_stems = _stem_sets_from_labels(labels_train_active)

    # Apply curriculum to training subset
    if curriculum == 'bucket_a_only':
        df_use = df_train[df_train.stem.isin(clean_stems_A)].reset_index(drop=True)
        print(f'  Curriculum A-only: {len(df_use)}/{len(df_train)}')
    elif curriculum == 'bucket_ab':
        df_use = df_train[df_train.stem.isin(active_stems)].reset_index(drop=True)
        print(f'  Curriculum A+B: {len(df_use)}/{len(df_train)}')
    else:
        df_use = df_train.reset_index(drop=True)

    if pseudo_df is not None and len(pseudo_df):
        df_use = pd.concat([df_use, pseudo_df], ignore_index=True)
        print(f'  + {len(pseudo_df)} pseudo-labeled samples → total {len(df_use)}')
        # Pseudo samples are TEST images — dataset needs TEST_IMG as alt search dir
        alt_dirs = [TEST_IMG]
        alt_roi_maps = [TEST_ROI]
    else:
        alt_dirs = None
        alt_roi_maps = None

    # Dataset: optionally with second-view tf for consistency
    tf2 = strong_tf if use_consistency else None
    tr_ds = BottleDataset(df_use, TRAIN_IMG, TRAIN_ROI, train_tf,
                          ann_cache=_MASK_CACHE if USE_GAIN else None,
                          second_view_tf=tf2,
                          alt_dirs=alt_dirs, alt_roi_maps=alt_roi_maps)
    va_ds = BottleDataset(df_val, TRAIN_IMG, TRAIN_ROI, valid_tf,
                          ann_cache=None)
    tr_ld = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True)
    va_ld = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    # Model — pretrained=True only when init_state is None (first run)
    model = BottleClsMT(drop_path_rate=drop_path_rate,
                        pretrained=(init_state is None)).to(PRIMARY_DEVICE)
    if init_state is not None:
        model.load_state_dict(init_state, strict=False)

    criterion = ComboLossNLS(alpha=0.25, gamma=2.0)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    total_steps = len(tr_ld) * epochs
    sched = WarmupCosineScheduler(opt, int(total_steps * 0.05), total_steps, lr)
    scaler = make_grad_scaler()
    ema = EMA(model, decay=0.999)

    epoch_states, epoch_f1 = [], []
    best_f1_in_phase = 0.0
    consistency_weight = CONSISTENCY_WEIGHT if use_consistency else 0.0

    for ep in range(1, epochs + 1):
        # Anneal GAIN omega from 0 → GAIN_OMEGA by mid-training
        omega = GAIN_OMEGA * min(1.0, ep / max(1, epochs // 2)) if anneal_gain else GAIN_OMEGA
        train_loss = train_epoch(model, tr_ld, opt, sched, scaler, ema, criterion,
                                 use_mix=use_mix, gain_omega=omega,
                                 consistency_w=consistency_weight)
        # EMA eval
        backup = {n: p.clone() for n, p in model.named_parameters()}
        ema.apply_to(model)
        oof_probs, oof_y, oof_bt, _ = eval_fold(model, va_ld)
        for n, p in model.named_parameters():
            if n in backup: p.data.copy_(backup[n])
        t, f = best_threshold(oof_y, oof_probs)
        print(f'  Ep {ep:2d}/{epochs}  loss={train_loss:.4f}  '
              f'EMA F1={f:.4f}@t={t:.3f}  gain_ω={omega:.2f}')
        epoch_states.append(copy.deepcopy(model.state_dict()))
        epoch_f1.append(f)
        torch.save(model.state_dict(),
                   SAVE / f'model_{VERSION_TAG}_{phase}_fold{fold}_lastep.pth')
        if f > best_f1_in_phase: best_f1_in_phase = f

    # SWA: avg 2 best epochs + last epoch
    if use_swa and len(epoch_states) >= 3:
        order = np.argsort(epoch_f1)[::-1]
        idxs = list(set(list(order[:2]) + [len(epoch_states)-1]))
        model.load_state_dict(average_state_dicts([epoch_states[i] for i in idxs]))
        if use_recal_bn:
            if recal_loader is not None:
                recal_bn_on_clean(model, recal_loader, device=DEVICE)
            else:
                # fallback: use current training loader (only safe pre-PL phase)
                recal_bn_on_clean(model, tr_ld, device=DEVICE)
        oof_probs, oof_y, oof_bt, _ = eval_fold(model, va_ld)
        t, f = best_threshold(oof_y, oof_probs)
        print(f'  SWA(recal_bn={use_recal_bn})  F1={f:.4f}@t={t:.3f}')

    torch.save(model.state_dict(),
               SAVE / f'model_{VERSION_TAG}_{phase}_fold{fold}.pth')
    np.save(SAVE / f'oof_{VERSION_TAG}_{phase}_fold{fold}.npy',     oof_probs)
    np.save(SAVE / f'oof_idx_{VERSION_TAG}_{phase}_fold{fold}.npy', np.array(df_val.index))
    np.save(SAVE / f'oof_bt_{VERSION_TAG}_{phase}_fold{fold}.npy',  oof_bt)
    return {'fold': fold, 'phase': phase, 'swa_f1': f, 'swa_t': t,
            'best_epoch_f1': best_f1_in_phase, 'model': model}


# Helper: build a CLEAN labeled loader for recal_bn (always bucket-A + B subset,
# NEVER pseudo-labeled). Used after PL phase SWA.
def make_clean_recal_loader(sample_frac=0.10):
    """A small clean labeled subset for BN recalibration."""
    n_sample = max(200, int(len(labels_train_active) * sample_frac))
    clean_subset = labels_train_active[labels_train_active.bucket != 'C'].sample(
        n=min(n_sample, (labels_train_active.bucket != 'C').sum()),
        random_state=SEED).reset_index(drop=True)
    ds = BottleDataset(clean_subset, TRAIN_IMG, TRAIN_ROI, valid_tf, ann_cache=None)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


print('train_fold() defined with curriculum + GAIN + NLS + consistency.')


In [ ]:
# =========================================================
# CELL 17: 🟢 PHASE 1 — folds 0-4 (curriculum: bucket-A only)
# =========================================================
# Fresh ImageNet-NS init, bucket-A only (cleanest labels), 8 epochs, SWA + recal_bn.

for fold in range(N_FOLDS):
    if fold_completed('p1', fold):
        print(f'p1 fold {fold} done — skip'); continue
    tr_idx, va_idx = SKF_SPLITS[fold]
    df_tr = labels_train_active.iloc[tr_idx].reset_index(drop=True)
    df_va = labels_train_active.iloc[va_idx].copy(); df_va.index = va_idx
    res = train_fold('p1', fold, df_tr, df_va,
                     P1_EPOCHS, LR_P1,
                     init_state=None, use_swa=True, use_recal_bn=True,
                     use_mix=True, curriculum='bucket_a_only',
                     drop_path_rate=DROP_PATH_RATE_P1)
    del res; gc.collect(); empty_device_cache()
    if fold == 2:
        print('\n>>> SAVE VERSION RECOMMENDED — Phase 1 folds 0-2 complete <<<')
print('\n>>> SAVE VERSION RECOMMENDED — Phase 1 complete <<<')
print('>>> Re-run cleanlab cell (cell 10) for informed bucket triage <<<')


In [ ]:
# =========================================================
# CELL 18: 🛡️ P1 backup submission (always have something submittable)
# =========================================================
test_df = pd.DataFrame({'image_id': test_ids})
sum_p, n_ok, bt_arr = None, 0, None
for fold in range(N_FOLDS):
    p = SAVE / f'model_{VERSION_TAG}_p1_fold{fold}.pth'
    if not p.exists(): continue
    m = BottleClsMT(drop_path_rate=DROP_PATH_RATE_P1, pretrained=False).to(PRIMARY_DEVICE)
    m.load_state_dict(torch.load(p, map_location=PRIMARY_DEVICE, weights_only=True))
    m.eval()
    probs, bts = predict_test_tta(m, test_df, TEST_IMG, TEST_ROI)
    sum_p = probs if sum_p is None else sum_p + probs
    bt_arr = bts; n_ok += 1
    del m; gc.collect(); empty_device_cache()

if n_ok > 0:
    avg = sum_p / n_ok
    t_dist = np.quantile(avg, 1 - labels_train_active.target.mean())
    sub = pd.DataFrame({'image_id': test_ids, 'target': (avg >= t_dist).astype(int)})
    sub.to_csv(SAVE / f'submission_{VERSION_TAG}_p1_backup.csv', index=False)
    np.save(SAVE / f'test_{VERSION_TAG}_p1_ensemble.npy', avg)
    np.save(SAVE / f'test_{VERSION_TAG}_btype.npy', bt_arr)
    print(f'P1 backup saved.  c1={(avg>=t_dist).mean():.4f}  t_dist={t_dist:.4f}')
else:
    print('No P1 weights found yet')


In [ ]:
# =========================================================
# CELL 19: Model-based noise cleaning at 0.95 confidence
# =========================================================
p1_oof = np.full(len(labels_train_active), np.nan, dtype=np.float32)
for fold in range(N_FOLDS):
    p = SAVE / f'oof_{VERSION_TAG}_p1_fold{fold}.npy'
    i = SAVE / f'oof_idx_{VERSION_TAG}_p1_fold{fold}.npy'
    if p.exists() and i.exists():
        idx = np.load(i); p1_oof[idx] = np.load(p)

valid = ~np.isnan(p1_oof)
y_curr = labels_train_active['target'].values
noisy  = np.zeros(len(labels_train_active), dtype=bool)
noisy[(y_curr == 0) & (p1_oof >= NOISE_CONF_THRESH) & valid] = True
noisy[(y_curr == 1) & (p1_oof <= 1-NOISE_CONF_THRESH) & valid] = True
n_noisy = noisy.sum()
print(f'Noise cleaning at conf≥{NOISE_CONF_THRESH}: {n_noisy} samples removed')

# Per-type breakdown
print('Per-type noise:')
for tname, tid in BOTTLE_TYPE_CANONICAL.items():
    tm = labels_train_active['btype_id'].values == tid
    nb = (noisy & tm).sum()
    print(f'  {tname}: {nb}/{tm.sum()}')

labels_clean = labels_train_active.loc[~noisy].reset_index(drop=True)
labels_clean.to_csv(SAVE / f'labels_clean_{VERSION_TAG}.csv', index=False)
clean_stems_set = set(labels_clean['stem'])
print(f'labels_clean: {len(labels_clean)} (dropped {n_noisy})')


In [ ]:
# =========================================================
# CELL 20: 🟡 PHASE 2 — folds 0-4 (curriculum: A+B, fine-tune on cleaned)
# =========================================================
for fold in range(N_FOLDS):
    if fold_completed('p2', fold):
        print(f'p2 fold {fold} done — skip'); continue
    tr_idx, va_idx = SKF_SPLITS[fold]
    df_va = labels_train_active.iloc[va_idx].copy(); df_va.index = va_idx
    df_tr_full = labels_train_active.iloc[tr_idx].reset_index(drop=True)
    df_tr = df_tr_full[df_tr_full.stem.isin(clean_stems_set)].reset_index(drop=True)

    init = torch.load(SAVE / f'model_{VERSION_TAG}_p1_fold{fold}.pth',
                      map_location=PRIMARY_DEVICE, weights_only=True)
    res = train_fold('p2', fold, df_tr, df_va,
                     P2_EPOCHS, LR_P2,
                     init_state=init, use_swa=True, use_recal_bn=True,
                     use_mix=True, curriculum='bucket_ab',  # NLS on bucket-B
                     drop_path_rate=DROP_PATH_RATE_P1)
    del res, init; gc.collect(); empty_device_cache()
    if fold == 2: print('\n>>> SAVE VERSION — Phase 2 folds 0-2 done <<<')

print('\n>>> SAVE VERSION — Phase 2 fully done <<<')


In [ ]:
# =========================================================
# CELL 21: 🎯 TEACHER EXPORT — per-fold P2 test predictions
# =========================================================
# This is the WHOLE POINT of the MaxViT notebook: produce ensemble-teacher
# predictions for the B4 v26 PL continuation. NO PL training, NO ONNX here.
#
# Saves test_lb_maxvit_p2_fold{0-4}.npy  (per-fold 6-mode TTA probabilities)
# and a fold-averaged test_lb_maxvit_p2_mean.npy for convenience.
# Also saves OOF holdout F1 so you can verify MaxViT trained well before
# trusting it as a teacher.

print('='*60)
print('ENSEMBLE MEMBER EXPORT - P2 test predictions')
print('='*60)

test_df = pd.DataFrame({'image_id': test_ids})

# ── 1. Per-fold test predictions (6-mode TTA) ───────────────
fold_test_preds = []
for fold in range(N_FOLDS):
    p = SAVE / f'model_{VERSION_TAG}_p2_fold{fold}.pth'
    if not p.exists():
        print(f'  fold {fold}: P2 weights missing — skip'); continue
    m = BottleClsMT(pretrained=False).to(PRIMARY_DEVICE)
    m.load_state_dict(torch.load(p, map_location=PRIMARY_DEVICE, weights_only=True), strict=False)
    m.eval()
    probs, bts = predict_test_tta(m, test_df, TEST_IMG, TEST_ROI)
    np.save(SAVE / f'test_{VERSION_TAG}_p2_fold{fold}.npy', probs)
    fold_test_preds.append(probs)
    print(f'  fold {fold}: saved test_{VERSION_TAG}_p2_fold{fold}.npy  '
          f'(mean prob={probs.mean():.4f})')
    del m; gc.collect(); empty_device_cache()

if fold_test_preds:
    mean_pred = np.mean(np.stack(fold_test_preds), axis=0)
    np.save(SAVE / f'test_{VERSION_TAG}_p2_mean.npy', mean_pred)
    print(f'\n  Saved fold-averaged: test_{VERSION_TAG}_p2_mean.npy  '
          f'(mean prob={mean_pred.mean():.4f}, n_folds={len(fold_test_preds)})')

# ── 2. OOF holdout F1 — verify MaxViT is a GOOD teacher ─────
# Reconstruct full-train OOF from saved P2 fold OOF files
p2_oof = np.full(len(labels_train_active), np.nan)
for fold in range(N_FOLDS):
    op = SAVE / f'oof_{VERSION_TAG}_p2_fold{fold}.npy'
    oi = SAVE / f'oof_idx_{VERSION_TAG}_p2_fold{fold}.npy'
    if op.exists() and oi.exists():
        idx = np.load(oi); p2_oof[idx] = np.load(op)
valid = ~np.isnan(p2_oof)
if valid.sum() > 0:
    y_active = labels_train_active['target'].values
    t_oof, f_oof = best_threshold(y_active[valid], p2_oof[valid])
    print(f'\n  MaxViT Tiny P2 OOF F1: {f_oof:.5f} @ t={t_oof:.3f}  '
          f'({valid.sum()}/{len(labels_train_active)} samples)')
    print(f'  → Compare to B4 v26 P2 OOF. If within ~0.01, MaxViT Tiny is a strong teacher.')

# ── 3. Holdout evaluation (true unseen check) ────────────────
try:
    labels_holdout_local = pd.read_csv(SAVE / f'labels_holdout_{VERSION_TAG}.csv')
    best_fold_p = SAVE / f'model_{VERSION_TAG}_p2_fold0.pth'
    if best_fold_p.exists() and len(labels_holdout_local):
        m = BottleClsMT(pretrained=False).to(PRIMARY_DEVICE)
        m.load_state_dict(torch.load(best_fold_p, map_location=PRIMARY_DEVICE,
                                     weights_only=True), strict=False)
        m.eval()
        hold_probs, hold_bts = predict_test_tta(
            m, labels_holdout_local[['image_id']], TRAIN_IMG, TRAIN_ROI)
        y_hold = labels_holdout_local['target'].values
        th, fh = best_threshold(y_hold, hold_probs)
        print(f'  MaxViT Tiny holdout F1 (fold0): {fh:.5f} @ t={th:.3f}')
        del m; gc.collect(); empty_device_cache()
except Exception as e:
    print(f'  Holdout eval skipped: {e}')

# ── 4. Upload instructions ───────────────────────────────────
print(f'''
{'='*60}
NEXT STEPS — wire MaxViT into B4 ensemble PL
{'='*60}
1. SAVE VERSION of this notebook (outputs persist).
2. Create a Kaggle Dataset from this notebook's output containing:
     test_{VERSION_TAG}_p2_fold0.npy ... fold4.npy
     test_{VERSION_TAG}_p2_mean.npy
3. In the B4 v26 PL-continuation notebook, ADD that dataset as input.
   The PL-gen cell already auto-detects files matching
   'test_*maxvit*p2*.npy' and blends 50/50 with B4's own P2 teacher.
4. The blended teacher generates better pseudo-labels → less overfit →
   higher final B4 F1. The FINAL ONNX stays the B4 model (fast, single-pass).

This notebook does NOT export ONNX. It exports fold and mean probabilities for ensembling.
{'='*60}''')
